# PennyLane fidelity quantum kernel

Construct a feature-map kernel from QNode state overlaps on default.qubit and MettleQ.

## What you will learn

- How to express this workflow with PennyLane's reference simulator.
- How to change only the execution target to MettleQ.
- How correctness is checked before comparing timings.
- How to decide whether this workload is large enough to benefit from Apple-native execution.

## Performance model: what is actually being compared?

The reference simulator is **already running on this Mac's Apple CPU**. MettleQ is not comparing Apple Silicon with a machine that ignores it. Its opportunity is to reduce state-evolution cost through its MLX/Metal path, while paying extra planning, adapter, dispatch, synchronization, and result-conversion overhead.

Consequently, small circuits should often be faster on the SDK reference. MettleQ becomes useful only when the simulated state or repeated workload is large enough to amortize that overhead. The final result says which path won this particular measurement; it never assumes MettleQ won.

## Imports and measurement helpers

The SDK imports define the circuit and reference simulator. MettleQ's adapter supplies the alternate backend/device. The shared helpers make timing and numerical checks identical across the suite.

In [1]:
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp

from mettleq.integrations.pennylane import MettleQDevice
from tutorials._support import (
    benchmark,
    emit_result,
    max_abs_error,
    pennylane_selection,
    phase_aligned_statevector_error,
    print_scaling_table,
    total_variation_distance,
)

## 1. Define the quantum problem

The kernel is built from overlaps between data-encoding quantum states.

In [2]:
data = np.asarray([[0.1, 0.2], [0.7, -0.4], [-0.5, 0.6], [0.2, 0.9]])

def make_state_qnode(device):
    @qml.qnode(device)
    def circuit(values):
        for wire in range(2):
            qml.Hadamard(wire)
            qml.RZ(values[wire], wires=wire)
        qml.IsingZZ(values[0] * values[1], wires=[0, 1])
        return qml.state()
    return circuit

def kernel(qnode):
    states = [np.asarray(qnode(row)) for row in data]
    return np.asarray([[abs(np.vdot(left, right)) ** 2 for right in states] for left in states])

reference_qnode = make_state_qnode(qml.device("default.qubit", wires=2))

## 2. Run and time the SDK reference

This is the baseline a user would normally run. `benchmark` performs an unmeasured warm-up, synchronizes lazy results, and reports the median of repeated complete calls—not just a selected kernel.

In [3]:
reference, reference_ms, _ = benchmark(lambda: kernel(reference_qnode))

## 3. Run the same problem with MettleQ

Only the execution target changes. MettleQ records whether it selected exact statevector or MPS and whether that method ran on CPU or GPU. The candidate result is timed under the same warm-up and repeat policy.

In [4]:
mettleq_device = MettleQDevice(wires=2, method="statevector", device="cpu")
mettleq_qnode = make_state_qnode(mettleq_device)
candidate, mettleq_ms, _ = benchmark(lambda: kernel(mettleq_qnode))
error = max_abs_error(reference, candidate)
method, device = pennylane_selection(mettleq_device)

## 4. Check correctness before discussing speed

The complete Gram matrix must agree numerically and retain its symmetry.

In [5]:
tutorial_result = emit_result(
    notebook="pennylane/09_quantum_kernel.ipynb",
    framework="pennylane",
    reference_ms=reference_ms,
    mettleq_ms=mettleq_ms,
    check="kernel matrix atol=4e-6",
    passed=error <= 4e-6,
    exact_match=bool(np.array_equal(reference, candidate)),
    selected_method=method,
    selected_device=device,
    metrics={"max_kernel_error": error, "reference": reference, "mettleq": candidate},
)


Comparison summary
------------------
Correctness contract: PASS — kernel matrix atol=4e-6
SDK reference median: 2.400 ms
MettleQ median:       3.230 ms
Timing interpretation: the SDK reference was 1.346x faster in this run.
MettleQ selected: statevector / cpu
Byte-for-byte result equality: no (see the declared tolerance/statistical check)

Machine-readable record (used by the suite runner):
TUTORIAL_RESULT::{"check": "kernel matrix atol=4e-6", "exact_match": false, "framework": "pennylane", "machine": "arm64", "metrics": {"max_kernel_error": 2.384185786574733e-07, "mettleq": [[0.9999997615814209, 0.814531147480011, 0.8544812798500061, 0.8745973110198975], [0.814531147480011, 0.9999997615814209, 0.5245655179023743, 0.5652012825012207], [0.8544812798500061, 0.5245655179023743, 0.9999997615814209, 0.8141177296638489], [0.8745973110198975, 0.5652012825012207, 0.8141177296638489, 0.9999997615814209]], "reference": [[0.9999999999999991, 0.8145313336586514, 0.8544814892633904, 0.87459737398

## What should you conclude?

This pattern can scale through batched state preparation; the compact example is an integration check.

Read the output in this order:

1. **Correctness contract** must pass. A fast wrong result is not useful.
2. **Reference / MettleQ ratio** above `1.0×` means MettleQ was faster; below `1.0×` means the SDK reference was faster.
3. **Selected method/device** explains whether MettleQ used statevector or MPS and CPU or GPU.
4. Treat this notebook as a reproducible observation on this Mac, not a universal performance claim.